# Module 9 — RAG Evaluation with RAGAS

RAGAS provides reference-free metrics to evaluate RAG pipelines.

## Key Metrics
| Metric | Measures | Range |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the context? | 0–1 |
| **Answer Relevance** | Does the answer address the question? | 0–1 |
| **Context Precision** | Are retrieved chunks relevant? | 0–1 |
| **Context Recall** | Are all needed facts retrieved? | 0–1 |
| **Noise Sensitivity** | Does irrelevant context hurt the answer? | 0–1 |

In [ ]:
# !pip install ragas langchain-openai

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# ── Build a sample dataset ────────────────────────────────────────────────────
samples = [
    SingleTurnSample(
        user_input="What is RAG?",
        response="RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a generative model.",
        retrieved_contexts=[
            "RAG (Retrieval-Augmented Generation) is a technique that retrieves relevant documents and passes them to an LLM.",
            "RAG was designed to reduce hallucination and provide up-to-date knowledge.",
        ],
        reference="RAG is a technique that retrieves relevant documents and uses them to ground LLM generation.",
    ),
    SingleTurnSample(
        user_input="What is the capital of France?",
        response="The capital of France is Berlin.",  # deliberate hallucination
        retrieved_contexts=[
            "Paris is the capital and largest city of France.",
            "France is a country in Western Europe.",
        ],
        reference="Paris is the capital of France.",
    ),
    SingleTurnSample(
        user_input="How does vector similarity search work?",
        response="Vector similarity search computes the cosine similarity between the query embedding and stored embeddings to find the most relevant documents.",
        retrieved_contexts=[
            "Vector stores index embeddings and use distance metrics like cosine similarity to find similar vectors.",
            "FAISS and Chroma are popular vector stores used for similarity search.",
        ],
        reference="Vector similarity search embeds the query and computes distances to stored vectors to retrieve the nearest neighbours.",
    ),
]

dataset = EvaluationDataset(samples=samples)
llm         = ChatOpenAI(model="gpt-4o-mini")
embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")

results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
    embeddings=embeddings,
)

print("\nRAGAS Evaluation Results:")
print("="*50)
df = results.to_pandas()
print(df[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].to_string())
print("\nMean scores:")
for col in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    print(f"  {col:25} : {df[col].mean():.4f}")


## Interpreting Results
- **Sample 2** (Paris/Berlin) should show low `faithfulness` — the answer contradicts the context.
- **Sample 1** should score high across all metrics.
- Use RAGAS scores to **compare retrieval strategies**, chunk sizes, or prompt templates.

In [ ]:
# ── Evaluate individual metric: Faithfulness breakdown ───────────────────────
from ragas.metrics import faithfulness
from ragas.dataset_schema import SingleTurnSample

# Faithfulness = # of answer claims supported by context / total claims
grounded_sample = SingleTurnSample(
    user_input="Who created Python?",
    response="Python was created by Guido van Rossum in the late 1980s.",
    retrieved_contexts=["Python was created by Guido van Rossum. Its implementation began in December 1989."],
    reference="Python was created by Guido van Rossum.",
)

ungrounded_sample = SingleTurnSample(
    user_input="Who created Python?",
    response="Python was created by Linus Torvalds in 2000 at Google headquarters.",
    retrieved_contexts=["Python was created by Guido van Rossum. Its implementation began in December 1989."],
    reference="Python was created by Guido van Rossum.",
)

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset

ds = EvaluationDataset(samples=[grounded_sample, ungrounded_sample])
r  = evaluate(dataset=ds, metrics=[faithfulness], llm=ChatOpenAI(model="gpt-4o-mini"),
              embeddings=OpenAIEmbeddings(model="text-embedding-3-small"))

df2 = r.to_pandas()
print("Faithfulness comparison:")
for i, row in df2.iterrows():
    label = "Grounded" if i == 0 else "Hallucinated"
    print(f"  {label}: faithfulness = {row['faithfulness']:.4f}")
